# Build Fashionpedia → Qdrant Index

Downloads images from Flickr, crops garments in memory, uploads to Supabase Storage, embeds via Gemini, upserts to Qdrant.

**Checkpoint/resume**: if Colab disconnects, reconnect and re-run from **Cell 1** onward — it picks up where it left off.

Phase 1 now saves progress incrementally: crops already in Supabase are skipped (no re-download), and the items cache + image checkpoint are saved to Drive every 100 images.

## 1. Install deps & mount Drive

In [ ]:
!pip install -q requests pillow

from google.colab import drive
drive.mount('/content/drive')

import os
ANNOTATIONS_PATH = "/content/drive/MyDrive/fashionpedia_train.json"
assert os.path.exists(ANNOTATIONS_PATH), f"Upload fashionpedia_train.json to your Google Drive root first!"
print(f"Found annotations: {os.path.getsize(ANNOTATIONS_PATH) / 1e6:.0f} MB")

## 2. Paste your API keys

In [ ]:
# Paste your keys here
QDRANT_URL = ""        # e.g. https://xxxxx.aws.cloud.qdrant.io:6333
QDRANT_API_KEY = ""
GEMINI_API_KEY = ""
SUPABASE_URL = ""      # e.g. https://xxxxx.supabase.co
SUPABASE_SERVICE_KEY = ""

assert all([QDRANT_URL, QDRANT_API_KEY, GEMINI_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_KEY]), "Fill in all keys above!"
print("Keys set ✓")

## 3. Run the build

This is the main cell. If Colab disconnects, reconnect, re-run cells 1 & 2, then re-run this — it resumes from checkpoint.

In [ ]:
import io
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from PIL import Image

# --------------- Config ---------------

COLLECTION = "fashionpedia_v2"
EMBED_DIM = 768
EMBED_BATCH = 96
EMBED_WORKERS = 4
UPSERT_BATCH = 500
CHECKPOINT_EVERY = 500
GEMINI_EMBED_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents?key={GEMINI_API_KEY}"

SUPABASE_BUCKET = "crops"
SUPABASE_HEADERS = {
    "Authorization": f"Bearer {SUPABASE_SERVICE_KEY}",
    "apikey": SUPABASE_SERVICE_KEY,
}

WEARABLE_IDS = set(range(27))

FASHIONPEDIA_ATTRIBUTES = {
    0: "floral", 1: "graphic", 2: "striped", 3: "plain", 4: "lattice",
    5: "spotted", 6: "checked", 7: "solid color",
    8: "denim", 9: "chiffon", 10: "corduroy", 11: "cotton",
    12: "faux fur", 13: "knit", 14: "lace", 15: "leather",
    16: "linen", 17: "mesh", 18: "nylon", 19: "satin",
    20: "sequined", 21: "silk", 22: "suede", 23: "velvet", 24: "wool",
    25: "long sleeve", 26: "short sleeve", 27: "sleeveless",
    28: "maxi length", 29: "midi length", 30: "mini length",
    31: "crew neckline", 32: "v-neckline", 33: "turtleneck",
    34: "sweetheart neckline", 35: "straight fit", 36: "loose fit", 37: "tight fit",
}

# Checkpoints on Drive so they survive disconnects
CHECKPOINT_PATH = Path("/content/drive/MyDrive/closet_drift_checkpoint.json")
ITEMS_CACHE_PATH = Path("/content/drive/MyDrive/closet_drift_items_cache.json")
PHASE1_CHECKPOINT_PATH = Path("/content/drive/MyDrive/closet_drift_phase1_checkpoint.json")

# --------------- Supabase Storage ---------------

def ensure_bucket():
    r = requests.get(f"{SUPABASE_URL}/storage/v1/bucket/{SUPABASE_BUCKET}", headers=SUPABASE_HEADERS, timeout=10)
    if r.status_code == 200:
        print(f"Supabase bucket '{SUPABASE_BUCKET}' exists")
        return
    resp = requests.post(
        f"{SUPABASE_URL}/storage/v1/bucket",
        headers={**SUPABASE_HEADERS, "Content-Type": "application/json"},
        json={"id": SUPABASE_BUCKET, "name": SUPABASE_BUCKET, "public": True},
        timeout=10,
    )
    resp.raise_for_status()
    print(f"Created Supabase bucket '{SUPABASE_BUCKET}'")

def list_supabase_crops():
    """List all crop filenames already in Supabase — used to skip re-uploads."""
    all_files = set()
    offset = 0
    limit = 1000
    while True:
        resp = requests.post(
            f"{SUPABASE_URL}/storage/v1/object/list/{SUPABASE_BUCKET}",
            headers={**SUPABASE_HEADERS, "Content-Type": "application/json"},
            json={"prefix": "", "limit": limit, "offset": offset},
            timeout=30,
        )
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        for f in batch:
            name = f.get("name", "")
            if name.endswith(".jpg"):
                all_files.add(name)
        if len(batch) < limit:
            break
        offset += limit
    return all_files

def upload_crop(filename, jpeg_bytes):
    resp = requests.post(
        f"{SUPABASE_URL}/storage/v1/object/{SUPABASE_BUCKET}/{filename}",
        headers={**SUPABASE_HEADERS, "Content-Type": "image/jpeg", "x-upsert": "true"},
        data=jpeg_bytes, timeout=30,
    )
    resp.raise_for_status()
    return f"{SUPABASE_URL}/storage/v1/object/public/{SUPABASE_BUCKET}/{filename}"

# --------------- Checkpoint ---------------

def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            data = json.loads(CHECKPOINT_PATH.read_text())
            n = data.get("items_done", 0)
            print(f"Resuming from checkpoint: {n} items already done")
            return n
        except Exception:
            pass
    return 0

def save_checkpoint(items_done):
    CHECKPOINT_PATH.write_text(json.dumps({"items_done": items_done, "collection": COLLECTION}))

def load_phase1_checkpoint():
    """Load Phase 1 checkpoint: which image index we processed up to."""
    if PHASE1_CHECKPOINT_PATH.exists():
        try:
            data = json.loads(PHASE1_CHECKPOINT_PATH.read_text())
            return data.get("images_done", 0)
        except Exception:
            pass
    return 0

def save_phase1_checkpoint(images_done):
    PHASE1_CHECKPOINT_PATH.write_text(json.dumps({"images_done": images_done}))

# --------------- Gemini Embedding ---------------

def _embed_one_batch(texts):
    body = {
        "requests": [
            {"model": "models/gemini-embedding-001", "content": {"parts": [{"text": t}]}, "outputDimensionality": EMBED_DIM}
            for t in texts
        ]
    }
    try:
        resp = requests.post(GEMINI_EMBED_URL, json=body, timeout=60)
        resp.raise_for_status()
        return [e["values"] for e in resp.json()["embeddings"]]
    except Exception:
        time.sleep(30)
        resp = requests.post(GEMINI_EMBED_URL, json=body, timeout=60)
        resp.raise_for_status()
        return [e["values"] for e in resp.json()["embeddings"]]

def embed_texts_parallel(texts):
    batches = [texts[i:i + EMBED_BATCH] for i in range(0, len(texts), EMBED_BATCH)]
    results = [[] for _ in batches]
    with ThreadPoolExecutor(max_workers=EMBED_WORKERS) as pool:
        fmap = {pool.submit(_embed_one_batch, b): i for i, b in enumerate(batches)}
        for future in as_completed(fmap):
            idx = fmap[future]
            try:
                results[idx] = future.result()
            except Exception as e:
                print(f"  [embed] batch {idx} failed: {e}")
                results[idx] = [[0.0] * EMBED_DIM] * len(batches[idx])
    return [v for batch in results for v in batch]

# --------------- Qdrant ---------------

def ensure_collection():
    r = requests.get(f"{QDRANT_URL}/collections/{COLLECTION}", headers={"api-key": QDRANT_API_KEY}, timeout=10)
    if r.status_code == 200:
        print(f"Collection '{COLLECTION}' exists")
        return
    resp = requests.put(
        f"{QDRANT_URL}/collections/{COLLECTION}",
        headers={"api-key": QDRANT_API_KEY, "Content-Type": "application/json"},
        json={"vectors": {"size": EMBED_DIM, "distance": "Cosine"}, "optimizers_config": {"indexing_threshold": 0}},
        timeout=10,
    )
    resp.raise_for_status()
    print(f"Created collection '{COLLECTION}'")

def upsert_batch(points):
    resp = requests.put(
        f"{QDRANT_URL}/collections/{COLLECTION}/points",
        headers={"api-key": QDRANT_API_KEY, "Content-Type": "application/json"},
        json={"points": points}, timeout=60,
    )
    resp.raise_for_status()

# --------------- Image Download + Crop ---------------

def download_image(original_url):
    if not original_url:
        return None
    try:
        resp = requests.get(original_url, timeout=30)
        resp.raise_for_status()
        return Image.open(io.BytesIO(resp.content))
    except Exception:
        return None

def crop_and_upload(img, bbox, ann_id, ann_img_w, ann_img_h, existing_crops, padding=0.1):
    """Crop garment from image, scaling bbox if image size differs from annotation."""
    crop_filename = f"{ann_id}.jpg"

    actual_w, actual_h = img.size
    scale_x = actual_w / ann_img_w if ann_img_w else 1
    scale_y = actual_h / ann_img_h if ann_img_h else 1

    x = bbox[0] * scale_x
    y = bbox[1] * scale_y
    w = bbox[2] * scale_x
    h = bbox[3] * scale_y

    pad_x, pad_y = w * padding, h * padding
    left = max(0, int(x - pad_x))
    top = max(0, int(y - pad_y))
    right = min(actual_w, int(x + w + pad_x))
    bottom = min(actual_h, int(y + h + pad_y))

    if right - left < 10 or bottom - top < 10:
        return None

    crop = img.crop((left, top, right, bottom))
    max_side = max(crop.size)
    if max_side > 400:
        scale = 400 / max_side
        crop = crop.resize((int(crop.size[0] * scale), int(crop.size[1] * scale)), Image.LANCZOS)

    crop_w, crop_h = crop.size

    # Skip upload if already in Supabase (batch-checked, no per-item HEAD request)
    if crop_filename in existing_crops:
        url = f"{SUPABASE_URL}/storage/v1/object/public/{SUPABASE_BUCKET}/{crop_filename}"
        return url, crop_w, crop_h

    buf = io.BytesIO()
    crop.save(buf, "JPEG", quality=85)
    try:
        url = upload_crop(crop_filename, buf.getvalue())
        return url, crop_w, crop_h
    except Exception as e:
        print(f"  [upload] {crop_filename} failed: {e}")
        return None

# --------------- Build Items ---------------

def build_items():
    # Load partial items cache if it exists
    items = []
    done_ann_ids = set()
    if ITEMS_CACHE_PATH.exists():
        try:
            with open(ITEMS_CACHE_PATH) as f:
                items = json.load(f)
            done_ann_ids = {it.get("ann_id") for it in items if it.get("ann_id")}
            print(f"Loaded {len(items)} items from partial cache")
        except Exception:
            items = []

    # Check Supabase for progress
    print("Checking Supabase for existing crops...")
    existing_crops = list_supabase_crops()
    print(f"Found {len(existing_crops)} crops already in Supabase")

    print("Loading annotations...")
    with open(ANNOTATIONS_PATH) as f:
        data = json.load(f)

    categories = {c["id"]: c["name"] for c in data["categories"]}
    attributes = {a["id"]: a["name"] for a in data["attributes"]}
    images_map = {img["id"]: img for img in data["images"]}

    img_anns = {}
    for ann in data["annotations"]:
        if ann["category_id"] in WEARABLE_IDS and ann.get("bbox"):
            img_anns.setdefault(ann["image_id"], []).append(ann)

    total_images = len(img_anns)
    image_ids = list(img_anns.keys())

    # Resume from Phase 1 checkpoint
    start_img_idx = load_phase1_checkpoint()
    if start_img_idx > 0:
        print(f"Resuming Phase 1 from image index {start_img_idx}/{total_images}")

    if start_img_idx >= total_images:
        print(f"Phase 1 already complete ({len(items)} items)")
        return items

    print(f"Processing images {start_img_idx}..{total_images} ({total_images - start_img_idx} remaining)...")

    skipped = 0
    processed = 0
    uploaded = 0
    reused = 0
    dl_batch = 20
    t0 = time.time()
    save_every = 100  # save items cache every N images

    for batch_start in range(start_img_idx, len(image_ids), dl_batch):
        batch_ids = image_ids[batch_start : batch_start + dl_batch]

        downloaded = {}
        with ThreadPoolExecutor(max_workers=8) as pool:
            futures = {}
            for img_id in batch_ids:
                img_info = images_map.get(img_id, {})
                url = img_info.get("original_url", "")
                if url:
                    futures[pool.submit(download_image, url)] = img_id
            for future in as_completed(futures):
                img_id = futures[future]
                result = future.result()
                if result:
                    downloaded[img_id] = result

        for img_id in batch_ids:
            img = downloaded.get(img_id)
            if not img:
                skipped += len(img_anns[img_id])
                continue

            img_info = images_map[img_id]
            ann_img_w = img_info.get("width", img.size[0])
            ann_img_h = img_info.get("height", img.size[1])

            for ann in img_anns[img_id]:
                if ann["id"] in done_ann_ids:
                    continue
                crop_result = crop_and_upload(img, ann["bbox"], ann["id"], ann_img_w, ann_img_h, existing_crops)
                if not crop_result:
                    skipped += 1
                    continue
                crop_url, crop_w, crop_h = crop_result
                if f"{ann['id']}.jpg" in existing_crops:
                    reused += 1
                else:
                    uploaded += 1
                cat_id = ann["category_id"]
                cat_name = categories.get(cat_id, "clothing")
                attr_ids = ann.get("attribute_ids", [])
                attr_names = [attributes.get(a, FASHIONPEDIA_ATTRIBUTES.get(a, "")) for a in attr_ids]
                attr_names = [a for a in attr_names if a]
                desc_parts = []
                if attr_names:
                    desc_parts.append(", ".join(attr_names))
                desc_parts.append(cat_name)
                items.append({
                    "id": len(items),
                    "ann_id": ann["id"],
                    "description": " ".join(desc_parts),
                    "category": cat_name,
                    "category_id": cat_id,
                    "attributes": attr_names,
                    "crop_url": crop_url,
                    "width": crop_w,
                    "height": crop_h,
                })

            img.close()
            processed += 1

        current_img_idx = batch_start + len(batch_ids)

        # Save incrementally to Drive
        if processed % save_every < dl_batch or current_img_idx >= len(image_ids):
            with open(ITEMS_CACHE_PATH, "w") as f:
                json.dump(items, f)
            save_phase1_checkpoint(current_img_idx)

        if processed % 100 < dl_batch and processed > 0:
            elapsed = time.time() - t0
            rate = processed / elapsed if elapsed > 0 else 0
            remaining_imgs = total_images - start_img_idx - processed
            eta = remaining_imgs / rate if rate > 0 else 0
            print(f"  [{start_img_idx + processed}/{total_images}] {uploaded} new + {reused} reused | {len(items)} total items | {skipped} skipped | {rate:.1f} img/s | ETA {eta/60:.0f}m")

    # Final save
    with open(ITEMS_CACHE_PATH, "w") as f:
        json.dump(items, f)
    save_phase1_checkpoint(total_images)

    print(f"Phase 1 done: {len(items)} items ({uploaded} new uploads, {reused} reused from Supabase, {skipped} skipped) in {(time.time()-t0)/60:.1f}m")
    return items

# --------------- Run it ---------------

print("=== Phase 0: Supabase bucket ===")
ensure_bucket()

print("\n=== Phase 1: Download + Crop + Upload ===")
items = build_items()
assert items, "No items built!"
print(f"\n{len(items)} items ready")

print("\n=== Phase 2: Qdrant collection ===")
ensure_collection()

print("\n=== Phase 3: Embed + Upsert ===")
start_from = load_checkpoint()
total = len(items)

if start_from >= total:
    print(f"All {total} items already indexed!")
else:
    if start_from > 0:
        print(f"Skipping first {start_from} items (already done)")

    remaining = items[start_from:]
    done = start_from
    t0 = time.time()

    for chunk_start in range(0, len(remaining), UPSERT_BATCH):
        chunk = remaining[chunk_start : chunk_start + UPSERT_BATCH]
        texts = [item["description"] for item in chunk]

        try:
            vectors = embed_texts_parallel(texts)
        except Exception as e:
            print(f"FATAL embed error at item {done}: {e}")
            save_checkpoint(done)
            print(f"Checkpoint saved at {done}. Re-run to resume.")
            raise

        points = []
        for item, vector in zip(chunk, vectors):
            points.append({
                "id": item["id"],
                "vector": vector,
                "payload": {
                    "description": item["description"],
                    "category": item["category"],
                    "category_id": item["category_id"],
                    "attributes": item["attributes"],
                    "crop_url": item["crop_url"],
                    "width": item["width"],
                    "height": item["height"],
                },
            })

        try:
            upsert_batch(points)
        except Exception as e:
            print(f"Upsert error at item {done}: {e}")
            save_checkpoint(done)
            print(f"Checkpoint saved at {done}. Re-run to resume.")
            raise

        done += len(chunk)

        if done % CHECKPOINT_EVERY < UPSERT_BATCH or chunk_start + UPSERT_BATCH >= len(remaining):
            save_checkpoint(done)

        elapsed = time.time() - t0
        rate = (done - start_from) / elapsed if elapsed > 0 else 0
        eta = (total - done) / rate if rate > 0 else 0
        print(f"  [{done}/{total}] embedded+upserted | {rate:.0f} items/s | ETA {eta/60:.0f}m")

    print(f"\nDone! {total} items indexed in {(time.time()-t0)/60:.1f}m")
    CHECKPOINT_PATH.unlink(missing_ok=True)
    PHASE1_CHECKPOINT_PATH.unlink(missing_ok=True)